In [39]:
import pandas as pd
import numpy as np

In [40]:
df=pd.read_csv(r"data/raw/Delhi_NCR_Delivery_Operations_Raw.csv")

In [41]:
df.shape

(30800, 22)

In [42]:
df.info

<bound method DataFrame.info of          Order_ID Customer_ID   Rider_ID      Dark_Store_ID  \
0      ORD_108266   CUST_8003  RIDER_188          DS_Dwarka   
1      ORD_103012   CUST_2730  RIDER_467           DS_Saket   
2      ORD_102210   CUST_8925  RIDER_517  DS_ConnaughtPlace   
3      ORD_104783   CUST_3208  RIDER_933  DS_ConnaughtPlace   
4      ORD_100142   CUST_2297  RIDER_600  DS_Gurugram_Sec29   
...           ...         ...        ...                ...   
30795  ORD_111472   CUST_1967  RIDER_512          DS_Dwarka   
30796  ORD_105008   CUST_9230  RIDER_503  DS_Gurugram_Sec29   
30797  ORD_129034   CUST_1873  RIDER_517  DS_Gurugram_Sec29   
30798  ORD_106670   CUST_8609  RIDER_675  DS_ConnaughtPlace   
30799  ORD_113343   CUST_4172  RIDER_293  DS_ConnaughtPlace   

                  Order_Timestamp       Weather Traffic_Density Vehicle_Type  \
0      2026-05-17 21:56:21.833233         Clear        Moderate   EV_Scooter   
1      2026-05-11 15:35:21.833233         Clear    

In [43]:
missing_values = df.isnull().sum()
missing_values

Order_ID                        0
Customer_ID                     0
Rider_ID                        0
Dark_Store_ID                   0
Order_Timestamp                 0
Weather                         0
Traffic_Density                 0
Vehicle_Type                    0
Primary_Category                0
Payment_Method                  0
Total_Items                     0
Delivery_Distance_KM            0
Order_Value_INR                 0
Is_Local_Rider                  0
Pack_Time_Mins                  0
Transit_Time_Mins               0
Total_Delivery_Time_Mins        0
SLA_Breached                    0
Legacy_Marketing_ID         30800
Temp_System_Log                 0
Customer_Phone_Raw           5751
Rider_Rating_Raw             5116
dtype: int64

In [44]:
missing_values = df.isna().sum().sort_values(ascending=False)
missing_values

Legacy_Marketing_ID         30800
Customer_Phone_Raw           5751
Rider_Rating_Raw             5116
Customer_ID                     0
Temp_System_Log                 0
SLA_Breached                    0
Total_Delivery_Time_Mins        0
Transit_Time_Mins               0
Pack_Time_Mins                  0
Is_Local_Rider                  0
Order_Value_INR                 0
Order_ID                        0
Total_Items                     0
Payment_Method                  0
Primary_Category                0
Vehicle_Type                    0
Traffic_Density                 0
Weather                         0
Order_Timestamp                 0
Dark_Store_ID                   0
Rider_ID                        0
Delivery_Distance_KM            0
dtype: int64

In [45]:
# 1. Drop irrelevant columns (Legacy_Marketing_ID is 100% null)
df = df.drop(columns=['Legacy_Marketing_ID', 'Temp_System_Log'])

# 2. Impute missing phone numbers with 'UNKNOWN'
df['Customer_Phone_Raw'] = df['Customer_Phone_Raw'].fillna('UNKNOWN')



In [46]:
# 3. Standardize Rider Ratings (Convert text strings to numeric floats)
rating_mapping = {
    'Terrible': 1.0,
    '2.0': 2.0,
    '3 out of 5': 3.0,
    '4.5 stars': 4.5,
    '5.0': 5.0
}

# Apply mapping to the original raw column
df['Rider_Rating_Raw'] = df['Rider_Rating_Raw'].map(rating_mapping)

# Impute remaining missing ratings with the median value of the numeric column
avg_rating = df['Rider_Rating_Raw'].median()
df['Rider_Rating_Raw'] = df['Rider_Rating_Raw'].fillna(avg_rating)

# Rename the column to a clean, professional name now that it is fixed
df = df.rename(columns={'Rider_Rating_Raw': 'Rider_Rating'})

# 4. Verify that all missing values have been resolved
print("Missing values after cleaning:")
print(df.isna().sum().sort_values(ascending=False))

Missing values after cleaning:
Order_ID                    0
Customer_ID                 0
Customer_Phone_Raw          0
SLA_Breached                0
Total_Delivery_Time_Mins    0
Transit_Time_Mins           0
Pack_Time_Mins              0
Is_Local_Rider              0
Order_Value_INR             0
Delivery_Distance_KM        0
Total_Items                 0
Payment_Method              0
Primary_Category            0
Vehicle_Type                0
Traffic_Density             0
Weather                     0
Order_Timestamp             0
Dark_Store_ID               0
Rider_ID                    0
Rider_Rating                0
dtype: int64


In [47]:
df.duplicated().sum()

np.int64(800)

In [48]:
# 1. Check for duplicate rows before cleaning
print(f"Duplicates before cleaning: {df.duplicated().sum()}")

# 2. Drop duplicate rows to ensure data integrity
df = df.drop_duplicates()

# 3. Verify final row count and duplicate status
print(f"Duplicates after cleaning: {df.duplicated().sum()}")
print(f"Final Total Rows: {df.shape[0]}")

Duplicates before cleaning: 800
Duplicates after cleaning: 0
Final Total Rows: 30000


In [49]:
# 1. Convert Order_Timestamp to a proper datetime object
df["Order_Timestamp"] = pd.to_datetime(df["Order_Timestamp"], errors="coerce")

# 2. Feature Engineering: Extract time-based dimensions for Dashboard analytics
df["Order_Date"] = df["Order_Timestamp"].dt.date
df["Order_Hour"] = df["Order_Timestamp"].dt.hour
df["Day_Name"] = df["Order_Timestamp"].dt.day_name()
df["Week"] = df["Order_Timestamp"].dt.isocalendar().week.astype("Int64")

# 3. Ensure critical columns have no missing data (Safety check)
critical_columns = ["Order_ID", "Order_Timestamp", "Dark_Store_ID", "SLA_Breached"]
df = df.dropna(subset=critical_columns)

# 4. Enforce strict data types to prevent SQL/Power BI import errors
df["Total_Items"] = df["Total_Items"].astype("int64")
df["Order_Value_INR"] = df["Order_Value_INR"].astype("float64")
df["SLA_Breached"] = df["SLA_Breached"].astype("int64")
df["Order_Hour"] = df["Order_Hour"].astype("int64")
df["Week"] = df["Week"].astype("int64")

# Verify the newly created temporal columns
print("Temporal features extracted successfully:")
print(df[["Order_Timestamp", "Order_Date", "Order_Hour", "Day_Name", "Week"]].head())

Temporal features extracted successfully:
             Order_Timestamp  Order_Date  Order_Hour   Day_Name  Week
0 2026-05-17 21:56:21.833233  2026-05-17          21     Sunday    20
1 2026-05-11 15:35:21.833233  2026-05-11          15     Monday    20
2 2026-05-06 20:35:21.833233  2026-05-06          20  Wednesday    19
3 2026-05-10 14:28:21.833233  2026-05-10          14     Sunday    19
4 2026-05-16 13:27:21.833233  2026-05-16          13   Saturday    20


 # Create Business Features

In [50]:


# 1. Advanced Metrics Calculation
# Calculate delivery speed (km/h) and handle division by zero (inf -> NaN)
df["Delivery_Speed_KMPH"] = (df["Delivery_Distance_KM"] / (df["Transit_Time_Mins"] / 60)).replace([np.inf, -np.inf], np.nan)

# Calculate unit economics (Revenue per KM)
df["Revenue_Per_KM"] = (df["Order_Value_INR"] / df["Delivery_Distance_KM"]).replace([np.inf, -np.inf], np.nan)

# 2. Strategic Bucketing for Dashboard Visualizations
# Create SLA Delivery Time cohorts
df["Delivery_Bucket"] = pd.cut(
    df["Total_Delivery_Time_Mins"],
    bins=[0, 10, 15, 20, 30, np.inf],
    labels=["0-10", "10-15", "15-20", "20-30", "30+"],
    right=False,
)

# Create Distance cohorts to analyze local vs long-distance efficiency
df["Distance_Bucket"] = pd.cut(
    df["Delivery_Distance_KM"],
    bins=[0, 1, 2, 3, 5, np.inf],
    labels=["0-1 km", "1-2 km", "2-3 km", "3-5 km", "5+ km"],
    right=False,
)

# Verify the newly engineered features
print("Feature Engineering completed:")
print(df[["Delivery_Speed_KMPH", "Revenue_Per_KM", "Delivery_Bucket", "Distance_Bucket"]].head())

Feature Engineering completed:
   Delivery_Speed_KMPH  Revenue_Per_KM Delivery_Bucket Distance_Bucket
0            24.000000       30.166667           15-20           5+ km
1            24.000000      921.631206           10-15          2-3 km
2            20.080680      236.865342           15-20          3-5 km
3            23.650626      617.431193           15-20          3-5 km
4            24.000000      271.129707            0-10          2-3 km


# Executive KPIs

In [51]:
# 1. Calculate High-Level Business KPIs
total_orders = len(df)
total_revenue = df["Order_Value_INR"].sum()
avg_order_value = df["Order_Value_INR"].mean()
avg_delivery_time = df["Total_Delivery_Time_Mins"].mean()

# Multiply by 100 to get percentage for SLA Breach
sla_breach_rate = df["SLA_Breached"].mean() * 100  

# Smart check: Use 'Rider_Rating' if it exists, else use 'Rider_Rating_Raw'
if "Rider_Rating" in df.columns:
    avg_rider_rating = df["Rider_Rating"].mean()
else:
    avg_rider_rating = df["Rider_Rating_Raw"].mean()

# 2. Create the KPI DataFrame with clear metric names
kpis = pd.DataFrame({
    "Metric": [
        "Total Orders", 
        "Total Revenue (INR)", 
        "Average Order Value (INR)", 
        "Average Delivery Time (Mins)", 
        "SLA Breach Rate (%)", 
        "Average Rider Rating"
    ],
    "Value": [
        total_orders, 
        total_revenue, 
        avg_order_value, 
        avg_delivery_time, 
        sla_breach_rate, 
        avg_rider_rating
    ]
})

# 3. Format the 'Value' column to look professional (2 decimal places)
kpis["Value"] = kpis["Value"].apply(lambda x: f"{x:,.2f}" if x % 1 != 0 else f"{int(x):,}")

# Display the Executive KPIs
print("--- QUICK-COMMERCE EXECUTIVE KPIs ---")
print(kpis.to_string(index=False))

--- QUICK-COMMERCE EXECUTIVE KPIs ---
                      Metric      Value
                Total Orders     30,000
         Total Revenue (INR) 54,653,038
   Average Order Value (INR)   1,821.77
Average Delivery Time (Mins)      12.70
         SLA Breach Rate (%)      54.46
        Average Rider Rating       3.06


# Store Performance Analysis

# Store Performance Analysis

In [52]:
# 1. Generate Store-Level Performance Metrics
# Grouping by Dark_Store_ID to analyze which specific zones are underperforming
store_performance = (
    df.groupby("Dark_Store_ID")
    .agg(
        Orders=("Order_ID", "count"),
        Revenue_INR=("Order_Value_INR", "sum"),
        Avg_Order_Value_INR=("Order_Value_INR", "mean"),
        Avg_Delivery_Time_Mins=("Total_Delivery_Time_Mins", "mean"),
        SLA_Breach_Rate_Pct=("SLA_Breached", "mean"),
        Avg_Distance_KM=("Delivery_Distance_KM", "mean"),
        Avg_Rider_Rating=("Rider_Rating", "mean"),
    )
    .reset_index()
    .sort_values("SLA_Breach_Rate_Pct", ascending=False)
)

# 2. Format columns for executive readability
# Convert SLA breach metric to a percentage format (0.54 -> 54%)
store_performance["SLA_Breach_Rate_Pct"] = store_performance["SLA_Breach_Rate_Pct"] * 100

# Round all numerical columns to 2 decimal places
store_performance = store_performance.round(2)

# Display the final Store Performance Table
print("--- DARK STORE PERFORMANCE ANALYSIS ---")
print(store_performance.to_string(index=False))

--- DARK STORE PERFORMANCE ANALYSIS ---
    Dark_Store_ID  Orders  Revenue_INR  Avg_Order_Value_INR  Avg_Delivery_Time_Mins  SLA_Breach_Rate_Pct  Avg_Distance_KM  Avg_Rider_Rating
        DS_Dwarka    5839   10754747.0              1841.88                   12.72                55.56             3.26              3.04
   DS_Noida_Sec18    6140   11187010.0              1821.99                   12.71                54.51             3.26              3.07
DS_Gurugram_Sec29    6039   10960643.0              1814.98                   12.72                54.43             3.26              3.08
         DS_Saket    5984   10882032.0              1818.52                   12.69                54.01             3.23              3.05
DS_ConnaughtPlace    5998   10868606.0              1812.04                   12.66                53.83             3.25              3.08


# Local Rider Impact Analysis

In [53]:
# 1. Analyze the Impact of Local Riders vs. Non-Local Riders
# Grouping by 'Is_Local_Rider' to measure the operational benefits of geographic familiarity
local_rider_impact = (
    df.groupby("Is_Local_Rider")
    .agg(
        Orders=("Order_ID", "count"),
        Revenue_INR=("Order_Value_INR", "sum"),
        Avg_Order_Value_INR=("Order_Value_INR", "mean"),
        Avg_Delivery_Time_Mins=("Total_Delivery_Time_Mins", "mean"),
        SLA_Breach_Rate_Pct=("SLA_Breached", "mean"),
        Avg_Distance_KM=("Delivery_Distance_KM", "mean"),
        Avg_Rider_Rating=("Rider_Rating", "mean"),
    )
    .reset_index()
)

# 2. Format the metrics for executive presentation
# Convert SLA breach from binary decimal to a readable percentage format
local_rider_impact["SLA_Breach_Rate_Pct"] = local_rider_impact["SLA_Breach_Rate_Pct"] * 100

# Round all numerical columns to 2 decimal places for a clean table
local_rider_impact = local_rider_impact.round(2)

# Display the Strategic Insight Table
print("--- LOCAL RIDER IMPACT ANALYSIS ---")
print(local_rider_impact.to_string(index=False))

--- LOCAL RIDER IMPACT ANALYSIS ---
Is_Local_Rider  Orders  Revenue_INR  Avg_Order_Value_INR  Avg_Delivery_Time_Mins  SLA_Breach_Rate_Pct  Avg_Distance_KM  Avg_Rider_Rating
            No   19539   35569378.0              1820.43                   13.20                58.01             3.25              3.07
           Yes   10461   19083660.0              1824.27                   11.77                47.83             3.25              3.05


In [54]:
# 1. Set the index to 'Is_Local_Rider' for easy row extraction
local = local_rider_impact.set_index("Is_Local_Rider")

# 2. Extract key metrics for Local (Yes) vs. Non-Local (No) riders
# Note: Using the updated column names from our previous formatting step
local_sla = local.loc["Yes", "SLA_Breach_Rate_Pct"]
non_local_sla = local.loc["No", "SLA_Breach_Rate_Pct"]
local_time = local.loc["Yes", "Avg_Delivery_Time_Mins"]
non_local_time = local.loc["No", "Avg_Delivery_Time_Mins"]

# 3. Print the Executive Summary Insights
print("--- KEY STRATEGIC TAKEAWAYS ---")
print(f"Local rider SLA breach rate: {local_sla:.2f}%")
print(f"Non-local rider SLA breach rate: {non_local_sla:.2f}%")

# Since the rates are already percentages, we just subtract them directly
print(f"SLA improvement using local riders: {(non_local_sla - local_sla):.2f} percentage points")
print(f"Average time saving using local riders: {(non_local_time - local_time):.2f} minutes per order")

--- KEY STRATEGIC TAKEAWAYS ---
Local rider SLA breach rate: 47.83%
Non-local rider SLA breach rate: 58.01%
SLA improvement using local riders: 10.18 percentage points
Average time saving using local riders: 1.43 minutes per order


# Hourly Demand Analysis

In [55]:
# 1. Analyze Peak Operational Hours
# Grouping by 'Order_Hour' to identify traffic spikes and SLA drops across the day
hourly_trend = (
    df.groupby("Order_Hour")
    .agg(
        Orders=("Order_ID", "count"),
        Revenue_INR=("Order_Value_INR", "sum"),
        Avg_Delivery_Time_Mins=("Total_Delivery_Time_Mins", "mean"),
        SLA_Breach_Rate_Pct=("SLA_Breached", "mean"),
    )
    .reset_index()
    .sort_values("Orders", ascending=False)
)

# 2. Format metrics for executive readability
# Convert SLA breach into a percentage
hourly_trend["SLA_Breach_Rate_Pct"] = hourly_trend["SLA_Breach_Rate_Pct"] * 100

# Round numerical columns to 2 decimal places
hourly_trend = hourly_trend.round(2)

# Ensure Order_Hour is displayed as a clean integer
hourly_trend["Order_Hour"] = hourly_trend["Order_Hour"].astype(int)

# 3. Display the Top 10 Busiest Hours
print("--- PEAK HOUR OPERATIONS ANALYSIS (TOP 10) ---")
print(hourly_trend.head(10).to_string(index=False))

--- PEAK HOUR OPERATIONS ANALYSIS (TOP 10) ---
 Order_Hour  Orders  Revenue_INR  Avg_Delivery_Time_Mins  SLA_Breach_Rate_Pct
          2    1320    2342759.0                   12.77                54.39
         20    1306    2370139.0                   12.62                54.67
          4    1305    2352516.0                   12.64                53.95
          5    1299    2358722.0                   12.73                53.35
         22    1299    2335591.0                   12.82                56.43
          8    1279    2324466.0                   12.71                55.12
         18    1270    2317824.0                   12.58                51.89
         12    1267    2335661.0                   12.70                56.12
         21    1265    2346183.0                   12.65                54.47
          3    1264    2360982.0                   12.65                54.19


# Delivery Root Cause Analysis

In [56]:
# 1. Root Cause Analysis: Operational Friction Points
# Grouping by external environmental factors and vehicle type to isolate SLA failures
delivery_drivers = (
    df.groupby(["Traffic_Density", "Weather", "Vehicle_Type"])
    .agg(
        Orders=("Order_ID", "count"),
        Avg_Delivery_Time_Mins=("Total_Delivery_Time_Mins", "mean"),
        SLA_Breach_Rate_Pct=("SLA_Breached", "mean"),
        Avg_Rider_Rating=("Rider_Rating", "mean"),
    )
    .reset_index()
    .sort_values(["SLA_Breach_Rate_Pct", "Orders"], ascending=[False, False])
)

# 2. Format metrics for executive readability
# Convert binary SLA breaches into a percentage format
delivery_drivers["SLA_Breach_Rate_Pct"] = delivery_drivers["SLA_Breach_Rate_Pct"] * 100

# Round all numerical columns to 2 decimal places
delivery_drivers = delivery_drivers.round(2)

# 3. Display the Top 10 Worst-Performing Scenarios
print("--- DELIVERY ROOT CAUSE ANALYSIS (TOP 10 WORST SCENARIOS) ---")
print(delivery_drivers.head(10).to_string(index=False))

--- DELIVERY ROOT CAUSE ANALYSIS (TOP 10 WORST SCENARIOS) ---
Traffic_Density      Weather Vehicle_Type  Orders  Avg_Delivery_Time_Mins  SLA_Breach_Rate_Pct  Avg_Rider_Rating
       Gridlock         Rain  Petrol_Bike     156                   21.20               100.00              3.15
       Gridlock         Rain      Bicycle      34                   21.07               100.00              3.04
       Gridlock          Fog      Bicycle      28                   18.69               100.00              3.55
       Gridlock         Rain   EV_Scooter     206                   20.93                99.03              3.21
       Gridlock Extreme_Heat   EV_Scooter     203                   19.07                99.01              3.15
       Gridlock Extreme_Heat  Petrol_Bike     176                   18.94                97.73              2.99
       Gridlock Extreme_Heat      Bicycle      43                   19.20                97.67              2.86
       Gridlock        Clear      

# Category Revenue Analysis

In [57]:
# 1. Product Category Performance Analysis
# Grouping by 'Primary_Category' to understand revenue drivers and category-specific SLA friction
category_performance = (
    df.groupby("Primary_Category")
    .agg(
        Orders=("Order_ID", "count"),
        Revenue_INR=("Order_Value_INR", "sum"),
        Avg_Order_Value_INR=("Order_Value_INR", "mean"),
        SLA_Breach_Rate_Pct=("SLA_Breached", "mean"),
        Avg_Delivery_Time_Mins=("Total_Delivery_Time_Mins", "mean"),
    )
    .reset_index()
    .sort_values("Revenue_INR", ascending=False)
)

# 2. Format metrics for executive readability
# Convert SLA breach into a percentage
category_performance["SLA_Breach_Rate_Pct"] = category_performance["SLA_Breach_Rate_Pct"] * 100

# Round numerical columns to 2 decimal places
category_performance = category_performance.round(2)

# 3. Display the Category Performance Table
print("--- PRODUCT CATEGORY PERFORMANCE ---")
print(category_performance.to_string(index=False))


--- PRODUCT CATEGORY PERFORMANCE ---
Primary_Category  Orders  Revenue_INR  Avg_Order_Value_INR  SLA_Breach_Rate_Pct  Avg_Delivery_Time_Mins
   Personal_Care    5079    9255733.0              1822.35                54.46                   12.65
          Snacks    5015    9192544.0              1833.01                54.56                   12.68
       Beverages    4969    9163334.0              1844.10                54.40                   12.68
           Dairy    4982    9038266.0              1814.18                54.28                   12.69
         Grocery    5038    9035400.0              1793.45                55.16                   12.75
     Paan_Corner    4917    8967761.0              1823.83                53.91                   12.74
